In [ ]:
import sys
sys.path.insert(0, "..")
from gensim.models import Word2Vec
from src.config import MODEL_PATH

model = Word2Vec.load(str(MODEL_PATH))


In [ ]:
from umap import UMAP
import matplotlib.pyplot as plt
import pandas as pd
from src.sessions import get_tracks

umap = UMAP(n_components = 2, n_neighbors = 15, min_dist = 0.1, metric = "cosine", random_state = 42)
umap_trans = umap.fit_transform(model.wv.vectors)


In [ ]:
tracks, play_count = get_tracks()
print(tracks.head())

from src.config import SESSIONS_CSV
streams = pd.read_csv(SESSIONS_CSV)
artist_streams = streams.groupby("artist_name").ms_played.sum().sort_values(ascending=False)
artists = pd.Series(tracks.loc[model.wv.index_to_key[i], "artist_name"] for i in range(len(model.wv.index_to_key)))
top_artists = artist_streams.head(10).index
is_top = pd.Series(a in top_artists for a in artists)
artist_color = dict(zip(top_artists, plt.cm.tab10.colors))

colors = [artist_color.get(a, "lightgrey") for a in artists]
plt.scatter(umap_trans[~is_top, 0], umap_trans[~is_top, 1], s=3, c = "lightgrey", label = "Other Artists")

for artist, color in artist_color.items():
    mask = artists == artist 
    plt.scatter(umap_trans[mask, 0], umap_trans[mask, 1], s=3, c = [color], label = artist)
plt.legend(fontsize = "small")
plt.show()

In [ ]:
from sklearn.cluster import HDBSCAN
clusterer = HDBSCAN(min_cluster_size=20)
labels = clusterer.fit_predict(umap_trans)

In [ ]:
print(len(set(labels)))
print(sum(labels == -1), sum(labels != -1))
cmap = plt.colormaps['gist_ncar'].resampled(52)
cluster_color = dict(zip(set(labels), [cmap(i) for i in range(52)]))

for cluster, color in cluster_color.items():
    mask = labels == cluster
    plt.scatter(umap_trans[mask, 0], umap_trans[mask, 1], s=3, c=[color])

# plt.legend(fontsize = "small")
plt.show()


In [ ]:
umap_3d = UMAP(n_components = 3, n_neighbors = 15, min_dist = 0.1, metric = "cosine", random_state = 42)
umap_trans_3d = umap_3d.fit_transform(model.wv.vectors)
labels_3d = clusterer.fit_predict(umap_trans_3d)
print(len(set(labels_3d)))

